# Step 5: Spatial domains and T-cell phenotypes across domains

Compute per-cell spatial-neighborhood compositions (radius 80 µm) and
cluster them with k-means to assign each cell to a spatial domain; domains
are then summarized as Immune-rich, Neuroblast-rich and Other. T-cell
subtype proportions and per-gene expression are compared across the three
domains.

CSV outputs in `data/processed/`:

- `fig4b.csv` (main-cell-type composition of each k-means domain)
- `fig4c_paired_wilcoxon.csv` (paired Wilcoxon tests of T-cell subtype proportion across domains)
- `fig4d_data.csv` (T-cell subtype by domain) and `ext_fig4b.csv` (per-sample T-cell subtype proportions per domain)
- `fig4d_friedman_stats.csv` (Friedman test across domains, per T-cell subtype)
- `fig4d_paired_wilcoxon_stats.csv` (paired Wilcoxon Immune-rich vs Neuroblast-rich, per gene)

The labeled AnnData is re-saved at the end with the `neigh_kmeans` and
`Spatial_Domain` annotations.


## Setup and imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import anndata as ad
import scimap as sm
from scipy import sparse
from scipy.stats import wilcoxon, friedmanchisquare
from statsmodels.stats.multitest import multipletests

sc.settings.verbosity = 3
np.random.seed(26)

indir   = '/path/to/integrated/processed_data/'
csvdir  = '../data/processed/'
figdir  = '../figures/'
os.makedirs(csvdir, exist_ok=True)
os.makedirs(figdir, exist_ok=True)

plt.rcParams['savefig.transparent'] = True
plt.rcParams['savefig.dpi'] = 600
mpl.rcParams['font.family'] = 'Helvetica'
mpl.rcParams['font.size'] = 6
sc.settings.figdir = figdir

# Main-celltype palette.
celltype_palette = {
    'Endothelial': '#d73027', 'Fibroblast': '#f46d43', 'Schwann': '#fdae61',
    'Neuroblast':  '#fee090', 'Macrophage': '#e0f3f8',
    'B':           '#abd9e9', 'T':          '#74add1',
}

# T-cell subtype palette sampled at five classes from matplotlib's YlGnBu
# colormap. Both Unicode and ASCII subtype spellings map to the same hex.
palette_map_T = {
    'Cytotoxic T':        '#ffffd9', 'Cytotoxic CD8T':     '#ffffd9',
    'Naive/CM T':         '#c7e9b4',
    'Proliferating T':    '#41b6c4', 'Proliferating CD8T': '#41b6c4',
    'Treg':               '#225ea8',
    'γδT':                '#081d58', 'gdT':                '#081d58',
}

## Load h5ad and compute spatial neighborhood domains

In [ ]:
adata = sc.read_h5ad(indir + 'xenium_integrated_labeled.h5ad')
adata.obs['X_centroid'] = adata.obsm['spatial'][:, 0]
adata.obs['Y_centroid'] = adata.obsm['spatial'][:, 1]

adata = sm.tl.spatial_count(
    adata, phenotype='celltypes_all', method='radius', radius=80,
    imageid='sample', x_coordinate='X_centroid', y_coordinate='Y_centroid',
    label='spatial_count',
)
adata = sm.tl.spatial_cluster(adata, df_name='spatial_count', method='kmeans', k=20, label='neigh_kmeans')

# Force natural numeric order for neigh_kmeans
nk = pd.to_numeric(adata.obs['neigh_kmeans'], errors='coerce')
order = sorted(nk.dropna().unique().astype(int))
adata.obs['neigh_kmeans'] = pd.Categorical(
    nk.astype('Int64').astype(str),
    categories=[str(i) for i in order], ordered=True,
)

## Main-cell-type composition of each spatial domain (Fig 4b)

In [ ]:
df = adata.obs[['neigh_kmeans', 'celltype']].copy()
df['neigh_kmeans'] = df['neigh_kmeans'].astype(str)
ct = df.groupby(['neigh_kmeans', 'celltype']).size().unstack(fill_value=0)
ct = ct.div(ct.sum(axis=1), axis=0).fillna(0)

desired_order = ['Neuroblast', 'Endothelial', 'Fibroblast', 'Schwann', 'Macrophage', 'B', 'T']
present = [c for c in desired_order if c in ct.columns]
extras  = [c for c in ct.columns if c not in present]
ct = ct[present + extras]

# Order rows by neuroblast proportion, then keep the remaining domains.
fixed_tail = ['8', '9', '0', '4']
neuro_prop = ct['Neuroblast'] if 'Neuroblast' in ct.columns else pd.Series(0.0, index=ct.index)
head_candidates = [k for k in ct.index if k not in fixed_tail]
head_sorted = neuro_prop.loc[head_candidates].sort_values(ascending=False).index.tolist()
tail_present = [k for k in fixed_tail if k in ct.index]
ct = ct.reindex(head_sorted + tail_present)

ct.reset_index().to_csv(csvdir + 'fig4b.csv', index=False)

In [ ]:
# Stacked bar of main-cell-type composition per spatial domain.
ct = pd.read_csv(csvdir + 'fig4b.csv').set_index('neigh_kmeans')

fig, ax = plt.subplots(figsize=(9/2.54, 8.5/2.54))
ct.plot.bar(stacked=True, color=[celltype_palette.get(c, '#E6E6E6') for c in ct.columns],
            edgecolor='none', linewidth=0, width=0.95, ax=ax, legend=False)
ax.set_ylim(0, 1.0)
ax.set_xlabel('Domain'); ax.set_ylabel('Proportion of Cells')
ax.set_xticks(np.arange(len(ct.index))); ax.set_xticklabels(ct.index, rotation=0, ha='right', fontsize=5)
for i, idx in enumerate(ct.index):
    bottom = 0.0
    for col in ct.columns:
        v = ct.loc[idx, col]
        if v >= 0.03:
            ax.text(i, bottom + v / 2, f'{v:.2f}', ha='center', va='center', fontsize=5, color='black')
        bottom += v
for side in ['top', 'right', 'left']:
    ax.spines[side].set_visible(False)
ax.spines['bottom'].set_visible(True)
plt.tight_layout()
plt.savefig(figdir + 'fig4b_domains.pdf', format='pdf', bbox_inches='tight')
plt.show(); plt.close(fig)

## Classify k-means domains into Immune-rich / Neuroblast-rich / Other

In [ ]:
immune_rich_clusters     = ['0', '4', '8', '9']
neuroblast_rich_clusters = ['1', '2', '6', '7', '10', '12', '14', '16', '17', '18']

def classify_3_domains(n):
    n_str = str(n)
    if n_str in immune_rich_clusters:
        return 'Immune-rich'
    if n_str in neuroblast_rich_clusters:
        return 'Neuroblast-rich'
    return 'Other'

t_cells = adata[adata.obs['celltype'] == 'T'].copy()
t_cells.obs['Spatial_Domain'] = t_cells.obs['neigh_kmeans'].apply(classify_3_domains)

# Also propagate to full adata (for downstream notebooks)
adata.obs['Spatial_Domain'] = adata.obs['neigh_kmeans'].apply(classify_3_domains)

## Paired Wilcoxon tests of T-cell subtype proportion across domains (Fig 4c)

Per-sample T-cell subtype proportions per spatial domain are tabulated
(`fig4c_tsubtype_domain_prop_by_sample.csv`), and paired Wilcoxon tests
across the three domains are reported in `fig4c_paired_wilcoxon.csv`.

In [ ]:
domain_order = ['Immune-rich', 'Neuroblast-rich', 'Other']

df = t_cells.obs[['sample', 'Spatial_Domain', 'T_subtype']].copy()
df = df[df['Spatial_Domain'].isin(domain_order)].copy()

ct_t = df.groupby(['sample', 'Spatial_Domain', 'T_subtype']).size().reset_index(name='count')
tot  = df.groupby(['sample', 'Spatial_Domain']).size().reset_index(name='total_t_in_domain')
prop = ct_t.merge(tot, on=['sample', 'Spatial_Domain'], how='left')
prop['prop'] = prop['count'] / prop['total_t_in_domain']
prop.to_csv(csvdir + 'fig4c_tsubtype_domain_prop_by_sample.csv', index=False)

subtypes = sorted(prop['T_subtype'].dropna().unique())
pairwise_domains = [
    ('Immune-rich', 'Neuroblast-rich'),
    ('Immune-rich', 'Other'),
    ('Neuroblast-rich', 'Other'),
]

pair_rows = []
for st in subtypes:
    dsub = prop[prop['T_subtype'] == st].copy()
    w = dsub.pivot_table(index='sample', columns='Spatial_Domain', values='prop', aggfunc='mean').reindex(columns=domain_order)
    for d1, d2 in pairwise_domains:
        wp = w[[d1, d2]].dropna().copy()
        n_pairs = len(wp)
        if n_pairs >= 3:
            try:
                stat_w, p_w = wilcoxon(wp[d1].values, wp[d2].values, alternative='two-sided', zero_method='wilcox', method='exact')
            except Exception:
                stat_w, p_w = wilcoxon(wp[d1].values, wp[d2].values, alternative='two-sided', zero_method='wilcox')
        else:
            stat_w, p_w = np.nan, np.nan
        pair_rows.append({
            'T_subtype': st, 'domain_1': d1, 'domain_2': d2,
            'n_pairs': int(n_pairs),
            'wilcoxon_W': float(stat_w) if pd.notna(stat_w) else np.nan,
            'p_value':    float(p_w)   if pd.notna(p_w)   else np.nan,
            'mean_1':   float(wp[d1].mean())   if n_pairs else np.nan,
            'mean_2':   float(wp[d2].mean())   if n_pairs else np.nan,
            'median_1': float(wp[d1].median()) if n_pairs else np.nan,
            'median_2': float(wp[d2].median()) if n_pairs else np.nan,
        })

pair_df = pd.DataFrame(pair_rows)
valid_p = pair_df['p_value'].notna()
if valid_p.any():
    _, qp, _, _ = multipletests(pair_df.loc[valid_p, 'p_value'].values, method='fdr_bh')
    pair_df.loc[valid_p, 'p_value_fdr_bh'] = qp
pair_df.to_csv(csvdir + 'fig4c_paired_wilcoxon.csv', index=False)
display(pair_df)

## T-cell subtype composition by spatial domain

Per-sample T-cell subtype proportions per domain are written to
`ext_fig4b.csv` together with a sample-weighted consensus row;
`fig4d_data.csv` contains only the consensus row.

In [ ]:
domain_order = ['Immune-rich', 'Neuroblast-rich', 'Other']

def build_ct(df_obs):
    df_t = df_obs[['Spatial_Domain', 'celltypes_all']].astype(str)
    ct_local = df_t.groupby(['Spatial_Domain', 'celltypes_all']).size().unstack(fill_value=0)
    ct_local = ct_local.div(ct_local.sum(axis=1), axis=0).fillna(0)
    ct_local = ct_local.reindex(domain_order)
    return ct_local.loc[ct_local.sum(axis=1) > 0]

source_rows = []
samples = sorted([s for s in t_cells.obs['sample'].dropna().unique()])
for sample in samples:
    t_sub = t_cells[t_cells.obs['sample'] == sample].copy()
    if t_sub.n_obs == 0:
        continue
    ct_sample = build_ct(t_sub.obs)
    if ct_sample.empty:
        continue
    long_df = (ct_sample.reset_index(names='Spatial_Domain')
               .melt(id_vars='Spatial_Domain', var_name='celltypes_all', value_name='Proportion'))
    long_df['Plot_Type'] = 'individual'
    long_df['Plot_ID']   = str(sample)
    source_rows.append(long_df)

ct_consensus = build_ct(t_cells.obs)
if not ct_consensus.empty:
    long_cons = (ct_consensus.reset_index(names='Spatial_Domain')
                 .melt(id_vars='Spatial_Domain', var_name='celltypes_all', value_name='Proportion'))
    long_cons['Plot_Type'] = 'consensus'
    long_cons['Plot_ID']   = 'consensus'
    source_rows.append(long_cons)

source_data = pd.concat(source_rows, ignore_index=True)
source_data['Spatial_Domain'] = pd.Categorical(source_data['Spatial_Domain'], categories=domain_order, ordered=True)
source_data = source_data[['Plot_Type', 'Plot_ID', 'Spatial_Domain', 'celltypes_all', 'Proportion']]\
    .sort_values(['Plot_Type', 'Plot_ID', 'Spatial_Domain', 'celltypes_all'])

source_data_wide = source_data.pivot_table(
    index=['Plot_Type', 'Plot_ID', 'Spatial_Domain'],
    columns='celltypes_all', values='Proportion', aggfunc='first', fill_value=0,
).reset_index()

source_data_wide.to_csv(csvdir + 'ext_fig4b.csv', index=False)
source_data_wide[source_data_wide['Plot_Type'] == 'consensus'].to_csv(csvdir + 'fig4d_data.csv', index=False)

### Stacked-bar T-cell subtype composition by spatial domain

One stacked bar per sample plus a consensus bar across samples. The x-axis
shows the three spatial domains (Immune-rich, Neuroblast-rich, Other) and
bar segments are T-cell subtype proportions.

In [ ]:
# Stacked-bar T-cell subtype composition per spatial domain.
from matplotlib.patches import Patch

comp = pd.read_csv(csvdir + 'ext_fig4b.csv')

# Align T-cell subtype column labels to the canonical Unicode forms.
comp = comp.rename(columns={
    'Cytotoxic CD8T':     'Cytotoxic T',
    'Proliferating CD8T': 'Proliferating T',
    'gdT':                'γδT',
})

# Stack order is alphabetical T-cell subtype: Cytotoxic T (light yellow, bottom)
# through γδT (dark blue, top).
subtype_order = ['Cytotoxic T', 'Naive/CM T', 'Proliferating T', 'Treg', 'γδT']
subtype_order = [s for s in subtype_order if s in comp.columns]

# Domain ordering on the x-axis: Other, Neuroblast-rich, Immune-rich.
domain_order_local = ['Other', 'Neuroblast-rich', 'Immune-rich']

def _label_color(hex_color):
    """Return '#FFFFFF' for dark fills, '#000000' for light, by perceptual luminance."""
    h = hex_color.lstrip('#')
    r, g, b = (int(h[i:i+2], 16) / 255 for i in (0, 2, 4))
    return '#FFFFFF' if (0.299*r + 0.587*g + 0.114*b) < 0.5 else '#000000'

for plot_id, df_g in comp.groupby('Plot_ID'):
    pivoted = (df_g.set_index('Spatial_Domain')[subtype_order]
                   .reindex(domain_order_local)
                   .fillna(0))
    fig, ax = plt.subplots(figsize=(4/2.54, 7/2.54))
    x = np.arange(len(domain_order_local))
    for i, domain in enumerate(domain_order_local):
        row = pivoted.loc[domain]
        bottom = 0.0
        for col in subtype_order:
            val = float(row[col])
            if val <= 0:
                continue
            ax.bar(x[i], val, bottom=bottom,
                   color=palette_map_T.get(col, '#E6E6E6'),
                   edgecolor='white', linewidth=0.5, width=0.8)
            if val >= 0.05:
                ax.text(x[i], bottom + val / 2, f'{val:.2f}',
                        ha='center', va='center', fontsize=5,
                        color=_label_color(palette_map_T.get(col, '#E6E6E6')))
            bottom += val
    ax.set_title(f'fig4d: {plot_id}')
    ax.set_xticks(x); ax.set_xticklabels(domain_order_local, rotation=90, ha='right')
    ax.set_ylim(0, 1.0); ax.set_xlim(-0.5, len(domain_order_local) - 0.5)
    ax.set_ylabel('Proportion')
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(axis='both', length=0)
    ax.grid(False)
    plt.tight_layout()
    plt.savefig(figdir + f'fig4d_Tsubtype_composition_{plot_id}.pdf',
                format='pdf', transparent=True, bbox_inches='tight')
    plt.show(); plt.close(fig)

## Friedman and paired Wilcoxon tests of per-gene expression across domains (Fig 4d)

Per-gene Friedman tests across the three spatial domains
(`fig4d_friedman_stats.csv`) and paired Wilcoxon tests of Immune-rich
versus Neuroblast-rich domains, per gene
(`fig4d_paired_wilcoxon_stats.csv`).

In [ ]:
# Friedman omnibus per T_subtype across the three domains (paired by sample)
domain_order = ['Immune-rich', 'Neuroblast-rich', 'Other']

prop = pd.read_csv(csvdir + 'fig4c_tsubtype_domain_prop_by_sample.csv')
subtypes = sorted(prop['T_subtype'].dropna().unique())

fried_rows = []
for st in subtypes:
    dsub = prop[prop['T_subtype'] == st].copy()
    w = dsub.pivot_table(index='sample', columns='Spatial_Domain', values='prop', aggfunc='mean').reindex(columns=domain_order)
    w_complete = w.dropna(subset=domain_order).copy()
    n_complete = len(w_complete)
    if n_complete >= 3:
        stat_f, p_f = friedmanchisquare(w_complete['Immune-rich'].values,
                                        w_complete['Neuroblast-rich'].values,
                                        w_complete['Other'].values)
    else:
        stat_f, p_f = np.nan, np.nan
    fried_rows.append({
        'T_subtype': st, 'n_complete_samples': int(n_complete),
        'friedman_chi2': float(stat_f) if pd.notna(stat_f) else np.nan,
        'friedman_p':    float(p_f)    if pd.notna(p_f)    else np.nan,
        'mean_Immune-rich':     float(w['Immune-rich'].mean())     if 'Immune-rich' in w     else np.nan,
        'mean_Neuroblast-rich': float(w['Neuroblast-rich'].mean()) if 'Neuroblast-rich' in w else np.nan,
        'mean_Other':           float(w['Other'].mean())           if 'Other' in w           else np.nan,
        'median_Immune-rich':     float(w['Immune-rich'].median())     if 'Immune-rich' in w     else np.nan,
        'median_Neuroblast-rich': float(w['Neuroblast-rich'].median()) if 'Neuroblast-rich' in w else np.nan,
        'median_Other':           float(w['Other'].median())           if 'Other' in w           else np.nan,
    })

fried_df = pd.DataFrame(fried_rows)
valid_f = fried_df['friedman_p'].notna()
if valid_f.any():
    _, qf, _, _ = multipletests(fried_df.loc[valid_f, 'friedman_p'].values, method='fdr_bh')
    fried_df.loc[valid_f, 'friedman_p_fdr_bh'] = qf
fried_df.to_csv(csvdir + 'fig4d_friedman_stats.csv', index=False)
display(fried_df)

In [ ]:
# Per-gene paired Wilcoxon: Immune-rich vs Neuroblast-rich (matches '4d stats paired wilcoxon')
domain_a, domain_b = 'Immune-rich', 'Neuroblast-rich'

genes_to_plot = [
    'CD4', 'CD8A', 'TRGC2',
    'CCR7', 'SELL', 'LEF1', 'TCF7', 'IL7R',
    'MKI67',
    'FOS', 'IFNG', 'GZMH', 'GZMB', 'GZMK', 'KLRD1',
    'PDCD1', 'LAG3', 'TOX', 'TIGIT',
]
genes = [g for g in genes_to_plot if g in t_cells.var_names]
if len(genes) == 0:
    raise ValueError('No genes from genes_to_plot found in t_cells.var_names')

X = t_cells[:, genes].X
if sparse.issparse(X):
    X = X.toarray()
expr_df = pd.DataFrame(X, columns=genes, index=t_cells.obs_names)
meta_t = t_cells.obs[['sample', 'Spatial_Domain']].copy()
dat = meta_t.join(expr_df)
dat2 = dat[dat['Spatial_Domain'].isin([domain_a, domain_b])].copy()

pb = dat2.groupby(['sample', 'Spatial_Domain'])[genes].mean().reset_index()

rows = []
for g in genes:
    wide = pb.pivot(index='sample', columns='Spatial_Domain', values=g).dropna(subset=[domain_a, domain_b])
    n_pairs = len(wide)
    if n_pairs < 3:
        rows.append({'gene': g, 'n_pairs': n_pairs,
                     'mean_Immune-rich': np.nan, 'mean_Neuroblast-rich': np.nan,
                     'median_delta_Neuro_minus_Immune': np.nan,
                     'wilcoxon_W': np.nan, 'p_value': np.nan})
        continue
    x = wide[domain_a].values; y = wide[domain_b].values
    delta = y - x
    try:
        W, p = wilcoxon(y, x, alternative='two-sided', zero_method='wilcox', method='exact')
    except Exception:
        W, p = wilcoxon(y, x, alternative='two-sided', zero_method='wilcox')
    rows.append({
        'gene': g, 'n_pairs': n_pairs,
        'mean_Immune-rich':     float(np.mean(x)),
        'mean_Neuroblast-rich': float(np.mean(y)),
        'median_delta_Neuro_minus_Immune': float(np.median(delta)),
        'wilcoxon_W': float(W), 'p_value': float(p),
    })

stats_df = pd.DataFrame(rows)
valid = stats_df['p_value'].notna()
if valid.any():
    _, qvals, _, _ = multipletests(stats_df.loc[valid, 'p_value'].values, method='fdr_bh')
    stats_df.loc[valid, 'p_value_fdr_bh'] = qvals

def q_to_stars(q):
    if pd.isna(q): return ''
    if q < 1e-3:   return '***'
    if q < 1e-2:   return '**'
    if q < 5e-2:   return '*'
    return ''
stats_df['stars'] = stats_df['p_value_fdr_bh'].apply(q_to_stars)
stats_df.to_csv(csvdir + 'fig4d_paired_wilcoxon_stats.csv', index=False)
display(stats_df.sort_values('p_value_fdr_bh', na_position='last'))

## Dotplot of T-cell gene expression across spatial domains

In [ ]:
dp = sc.pl.dotplot(
    t_cells, genes_to_plot, groupby='Spatial_Domain',
    categories_order=['Immune-rich', 'Neuroblast-rich', 'Other'],
    cmap='YlGnBu', return_fig=True, show=False, standard_scale='var',
)
dp = dp.style(smallest_dot=0.01, largest_dot=70)
fig = dp.fig
fig.set_size_inches(9/2.54, 3.5/2.54, forward=True)
fig.tight_layout()
fig.savefig(figdir + 'fig4d_dotplot_T_domains.pdf', bbox_inches='tight')
plt.close(fig)

## Save labeled AnnData with spatial-domain annotations

In [ ]:
adata.write_h5ad(indir + 'xenium_integrated_labeled.h5ad', compression='gzip')